# Классификация: оригинальный трейн vs аугментированный

Единый ноутбук. Прогоняет один и тот же стенд (TF-IDF + 4 BERT-модели)
на двух датасетах подряд и строит сравнительную визуализацию.

**Входы:** `train.csv` (оригинал), `train_final.csv` (аугментированный), `test.csv`
**Выход:** таблица результатов + сравнительный график.

## 1. Установка зависимостей

In [ ]:
%%capture
!pip install transformers torch scikit-learn pandas matplotlib

In [ ]:
import torch
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Конфигурация

Пути к датасетам и список экспериментов. LabelEncoder фитим на ОБЪЕДИНЕНИИ
меток train+test (метки одни и те же), чтобы кодировка совпадала для обоих прогонов.

In [ ]:
import random
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    classification_report,
)

# ── Пути ────────────────────────────────────────────────────────────────────
ORIG_TRAIN_PATH = Path('/content/train.csv')        # оригинал без аугментации
AUG_TRAIN_PATH  = Path('/content/train_final.csv')  # аугментированный
TEST_PATH       = Path('/content/test.csv')
RESULTS_PATH    = Path('/content/experiments_results.csv')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Тестовая выборка одна для обоих прогонов ────────────────────────────────
test_df = pd.read_csv(TEST_PATH)

# ── LabelEncoder фитим на метках теста (полный набор классов) ───────────────
label_encoder = LabelEncoder()
label_encoder.fit(test_df['label'])
NUM_LABELS = len(label_encoder.classes_)
test_df['label_id'] = label_encoder.transform(test_df['label'])

print(f'Test: {len(test_df)} документов')
print(f'Классов: {NUM_LABELS}')

# ── Накопители ──────────────────────────────────────────────────────────────
all_results = []
per_class_reports = {}  # ключ: (dataset_tag, short_name)

## 3. Список BERT-экспериментов

In [ ]:
BERT_EXPERIMENTS = [
    {
        'model_name': 'cointegrated/rubert-tiny2',
        'short_name': 'rubert-tiny2',
        'max_length': 512,
        'batch_size': 16,
        'epochs': 5,
        'learning_rate': 1e-5,
    },
    {
        'model_name': 'DeepPavlov/rubert-base-cased',
        'short_name': 'rubert-base',
        'max_length': 512,
        'batch_size': 8,
        'epochs': 5,
        'learning_rate': 2e-5,
    },
    {
        'model_name': 'ai-forever/ruRoberta-large',
        'short_name': 'ruroberta-large',
        'max_length': 512,
        'batch_size': 4,
        'epochs': 5,
        'learning_rate': 1e-5,
    },
    {
        'model_name': 'deepvk/RuModernBERT-base',
        'short_name': 'rumodernbert-base',
        'max_length': 512,
        'batch_size': 8,
        'epochs': 5,
        'learning_rate': 2e-5,
    },
]

## 4. Вспомогательные функции для BERT

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


def run_bert_experiment(config, train_df, test_df, label_encoder, num_labels, dataset_tag):
    print(f'ЭКСПЕРИМЕНТ: {config["short_name"]}')
    print(f'Модель: {config["model_name"]}')


    tokenizer = AutoTokenizer.from_pretrained(config['model_name'])

    def tokenize(texts):
        return tokenizer(
            texts,
            max_length=config['max_length'],
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )

    train_enc = tokenize(train_df['text'].tolist())
    test_enc = tokenize(test_df['text'].tolist())

    train_ds = TextDataset(train_enc, train_df['label_id'].values)
    test_ds = TextDataset(test_enc, test_df['label_id'].values)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=config['batch_size'], shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        config['model_name'],
        num_labels=num_labels,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Параметров: {n_params:.1f}M')

    optimizer = AdamW(model.parameters(), lr=config['learning_rate'])
    total_steps = len(train_loader) * config['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=100, num_training_steps=total_steps
    )

    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0
        progress = tqdm(train_loader, desc=f'Эпоха {epoch+1}/{config["epochs"]}')
        for batch in progress:
            optimizer.zero_grad()
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device),
            )
            outputs.loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += outputs.loss.item()
            progress.set_postfix(loss=outputs.loss.item())
        print(f'  Эпоха {epoch+1}: средний loss = {epoch_loss / len(train_loader):.4f}')

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Оценка'):
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
            )
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].numpy())

    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)

    print(f'\nРезультаты:')
    print(f'  Balanced accuracy: {bal_acc:.4f}')
    print(f'  Macro F1:          {macro_f1:.4f}')
    print(f'  Accuracy:          {acc:.4f}')

    report = classification_report(
        all_labels, all_preds,
        target_names=label_encoder.classes_,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T.round(4)

    del model, tokenizer, train_enc, test_enc, train_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'short_name': config['short_name'],
        'model_name': config['model_name'],
        'n_params_M': round(n_params, 1),
        'max_length': config['max_length'],
        'batch_size': config['batch_size'],
        'epochs': config['epochs'],
        'learning_rate': config['learning_rate'],
        'dataset': dataset_tag,
        'train_size': len(train_df),
        'balanced_accuracy': round(bal_acc, 4),
        'macro_f1': round(macro_f1, 4),
        'accuracy': round(acc, 4),
    }, report_df

## 5. TF-IDF + Логистическая регрессия (функция)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def run_tfidf(train_df, test_df, dataset_tag):
    print(f'TF-IDF + LogReg | датасет: {dataset_tag}')
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,
            analyzer='word',
        )),
        ('clf', LogisticRegression(
            max_iter=1000,
            C=5.0,
            class_weight='balanced',
            solver='saga',
            n_jobs=-1,
            random_state=SEED,
        )),
    ])
    pipeline.fit(train_df['text'], train_df['label'])
    preds = pipeline.predict(test_df['text'])

    bal_acc = balanced_accuracy_score(test_df['label'], preds)
    macro_f1 = f1_score(test_df['label'], preds, average='macro', zero_division=0)
    acc = accuracy_score(test_df['label'], preds)

    print(f'  Balanced accuracy: {bal_acc:.4f}')
    print(f'  Macro F1:          {macro_f1:.4f}')
    print(f'  Accuracy:          {acc:.4f}')

    report = classification_report(test_df['label'], preds, zero_division=0, output_dict=True)
    report_df = pd.DataFrame(report).T.round(4)
    per_class_reports[(dataset_tag, 'tfidf-logreg')] = report_df

    return {
        'short_name': 'tfidf-logreg',
        'model_name': 'TF-IDF + LogisticRegression',
        'n_params_M': '-',
        'max_length': '-',
        'batch_size': '-',
        'epochs': '-',
        'learning_rate': '-',
        'dataset': dataset_tag,
        'train_size': len(train_df),
        'balanced_accuracy': round(bal_acc, 4),
        'macro_f1': round(macro_f1, 4),
        'accuracy': round(acc, 4),
    }

## 6. Раннер: все модели на одном датасете

Функция прогоняет TF-IDF + все BERT-модели на переданном train и
дописывает результаты в общий список с пометкой `dataset_tag`.

In [ ]:
def run_all_models(train_path, dataset_tag):
    print('=' * 70)
    print(f'ДАТАСЕТ: {dataset_tag}  ({train_path})')
    print('=' * 70)

    train_df = pd.read_csv(train_path)
    train_df['label_id'] = label_encoder.transform(train_df['label'])
    print(f'Train: {len(train_df)} документов\n')

    # TF-IDF baseline
    all_results.append(run_tfidf(train_df, test_df, dataset_tag))

    # BERT-модели
    for config in BERT_EXPERIMENTS:
        result, report_df = run_bert_experiment(
            config, train_df, test_df, label_encoder, NUM_LABELS, dataset_tag
        )
        all_results.append(result)
        per_class_reports[(dataset_tag, config['short_name'])] = report_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f'\nДатасет {dataset_tag} — готово\n')

## 7. Запуск на обоих датасетах

Сначала оригинал, потом аугментированный. Это долго — 2× по 5 моделей.

In [ ]:
run_all_models(ORIG_TRAIN_PATH, 'Оригинальный')
run_all_models(AUG_TRAIN_PATH, 'Аугментированный')

## 8. Сводная таблица результатов

In [ ]:
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_PATH, index=False)

comparison = results_df[[
    'short_name', 'dataset', 'train_size',
    'balanced_accuracy', 'macro_f1', 'accuracy'
]].sort_values(['short_name', 'dataset'])
print(comparison.to_string(index=False))

## 9. Визуализация: оригинальный трейн vs аугментированный

Две панели (Balanced Accuracy и Macro F1), синий = оригинал, зелёный = аугментированный.

In [ ]:
import matplotlib.pyplot as plt

MODEL_ORDER = ['tfidf-logreg', 'rubert-tiny2', 'rubert-base', 'ruroberta-large', 'rumodernbert-base']
MODEL_LABELS = {
    'tfidf-logreg':      'TF-IDF\n+LogReg',
    'rubert-tiny2':      'rubert-\ntiny2',
    'rubert-base':       'rubert-\nbase',
    'ruroberta-large':   'ruRoberta-\nlarge',
    'rumodernbert-base': 'RuModern-\nBERT-base',
}

def get_metric(dataset_tag, model, metric):
    row = results_df[(results_df['dataset'] == dataset_tag) & (results_df['short_name'] == model)]
    return row[metric].values[0] if len(row) else np.nan

orig_size = int(results_df[results_df['dataset'] == 'Оригинальный']['train_size'].iloc[0])
aug_size  = int(results_df[results_df['dataset'] == 'Аугментированный']['train_size'].iloc[0])

COLOR_ORIG = '#5B7FB4'   # синий
COLOR_AUG  = '#6FAE7A'   # зелёный

labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
x = np.arange(len(MODEL_ORDER))
width = 0.38

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Сравнение моделей: оригинальный трейн vs аугментированный',
             fontsize=15, fontweight='bold', y=1.02)

for ax, metric, title in [
    (axes[0], 'balanced_accuracy', 'Balanced Accuracy'),
    (axes[1], 'macro_f1', 'Macro F1'),
]:
    orig_vals = [get_metric('Оригинальный', m, metric) for m in MODEL_ORDER]
    aug_vals  = [get_metric('Аугментированный', m, metric) for m in MODEL_ORDER]

    bars1 = ax.bar(x - width/2, orig_vals, width,
                   label=f'Оригинальный трейн ({orig_size})', color=COLOR_ORIG)
    bars2 = ax.bar(x + width/2, aug_vals, width,
                   label=f'Аугментированный ({aug_size})', color=COLOR_AUG)

    for bars in (bars1, bars2):
        for b in bars:
            h = b.get_height()
            if not np.isnan(h):
                ax.text(b.get_x() + b.get_width()/2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=9)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Значение метрики', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 0.65)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('/content/comparison_orig_vs_aug.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: comparison_orig_vs_aug.png')

## 10. Анализ худших классов (аугментированный датасет)

In [ ]:
for (dataset_tag, name), report in per_class_reports.items():
    if dataset_tag != 'Аугментированный':
        continue
    print(f'\nХудшие 10 классов по F1 [{name}]:')
    classes = report.loc[~report.index.isin(['accuracy', 'macro avg', 'weighted avg'])]
    print(classes.sort_values('f1-score').head(10)[['precision', 'recall', 'f1-score', 'support']])